VINOD KUMAR K V

AI-powered paraphrasing tool

Develop an AI-powered paraphrasing tool as a Python application or module. The tool should:

● Take a block of input text

● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar via Hugging Face Transformers)

● Generate a paraphrased version preserving meaning, improving clarity, and ensuring originality.

● Include built-in checks for grammar, spelling, and fluency of output

Recommended technologies:

● Python, with Hugging Face Transformers and relevant NLP packages (spaCy, NLTK, etc.)

● TensorFlow or PyTorch for any model fine-tuning

● Use only console or script-based input/output (no GUI or web interface)

● Optional: Integrate evaluation metrics like BLEU, ROUGE, or semantic similarity scores

Expected Output:

● Well-commented Python scripts/notebooks

● Sample results demonstrating paraphrased outputs

● Evaluation report showing tool accuracy and originality

In [1]:
#Cell 01
# Install necessary libraries for transformers, evaluation, and grammar checking
!pip install -q transformers torch sentencepiece
!pip install -q language_tool_python spacy
!pip install -q evaluate rouge_score sentence_transformers
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
#Cell 02

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
import evaluate
from sentence_transformers import SentenceTransformer, util
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Initialize Grammar & Spelling Checker
# LanguageTool checks for spelling, grammar, and fluency issues
grammar_tool = language_tool_python.LanguageTool('en-US')

# Initialize Evaluation Metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# Initialize Sentence Transformer for Semantic Similarity
# This model converts sentences to vectors to check if the meaning was preserved
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

print("All libraries imported and tools initialized successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All libraries imported and tools initialized successfully!


In [3]:
#Cell 03

# Load Tokenizer and Model from Hugging Face
model_name = "Vamsi/T5_Paraphrase_Paws"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU if available in Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded successfully on {device}!")

Loading Vamsi/T5_Paraphrase_Paws...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model loaded successfully on cpu!


In [4]:
# Cell 4
def correct_grammar(text):
    """Checks and corrects grammar/spelling using LanguageTool."""
    matches = grammar_tool.check(text)
    corrected_text = language_tool_python.utils.correct(text, matches)
    return corrected_text, len(matches)

def generate_paraphrase(sentence, num_return_sequences=1):
    """Generates paraphrases using the T5 model."""
    # T5 requires a specific prompt format for fine-tuned tasks
    text = "paraphrase: " + sentence + " </s>"

    # FIX: Use the modern tokenizer direct call instead of encode_plus
    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    # Generate output using beam search for better quality
    outputs = model.generate(
        input_ids=encoding['input_ids'],
        attention_mask=encoding['attention_mask'],
        max_length=256,
        do_sample=True,           # Turns on randomness/creativity
        top_k=120,                # Broadens the vocabulary choices
        top_p=0.95,               # Nucleus sampling
        temperature=0.9,          # Higher temp = more diverse (0.0 to 1.0+)
        repetition_penalty=1.2,   # Penalizes repeating the exact same words
        num_return_sequences=num_return_sequences
    )

    results = []
    for output in outputs:
        # Decode the generated text
        paraphrased_text = tokenizer.decode(output, skip_special_tokens=True, clean_up_tokenization_spaces=True)

        # Apply grammar, spelling, and fluency check
        final_text, errors_fixed = correct_grammar(paraphrased_text)
        results.append({
            "text": final_text,
            "grammar_fixes_applied": errors_fixed
        })

    return results[0] # Return the top result

def calculate_similarity(original, paraphrased):
    """Calculates Cosine Similarity to ensure meaning is preserved."""
    embeddings1 = similarity_model.encode(original, convert_to_tensor=True)
    embeddings2 = similarity_model.encode(paraphrased, convert_to_tensor=True)
    cosine_scores = util.cos_sim(embeddings1, embeddings2)
    return cosine_scores.item()

In [5]:
#Cell 05

# Sample input texts
input_texts = [
    "The talented chef cooked a delicious meal for all the guests.",
    "It is highly recommended to drink plenty of water every single day.",
    "A large number of people enjoy taking a relaxing walk in the park during the spring.",
    "The professor carefully explained the difficult problem to the confused students."
]

paraphrased_outputs = []

print("--- PARAPHRASING RESULTS ---\n")
for original in input_texts:
    result = generate_paraphrase(original)
    paraphrased = result["text"]
    paraphrased_outputs.append(paraphrased)

    similarity = calculate_similarity(original, paraphrased)

    print(f"Original:    {original}")
    print(f"Paraphrased: {paraphrased}")
    print(f"Grammar Fixes: {result['grammar_fixes_applied']}")
    print(f"Semantic Similarity: {similarity:.4f}\n")

--- PARAPHRASING RESULTS ---

Original:    The talented chef cooked a delicious meal for all the guests.
Paraphrased: The talented chef cooked a delicious meal for all guests.
Grammar Fixes: 0
Semantic Similarity: 0.9956

Original:    It is highly recommended to drink plenty of water every single day.
Paraphrased: It is highly recommended to drink plenty of water every single day.
Grammar Fixes: 0
Semantic Similarity: 1.0000

Original:    A large number of people enjoy taking a relaxing walk in the park during the spring.
Paraphrased: During spring many people enjoy a relaxing walk in the park.
Grammar Fixes: 2
Semantic Similarity: 0.9573

Original:    The professor carefully explained the difficult problem to the confused students.
Paraphrased: The professor explained the difficult problem to the confused students carefully.
Grammar Fixes: 0
Semantic Similarity: 0.9962



In [6]:
#Cell 06

print("--- EVALUATION REPORT ---\n")

# Calculate ROUGE score (Measures overlap. Lower overlap = higher originality)
rouge_results = rouge_metric.compute(predictions=paraphrased_outputs, references=input_texts)

# Calculate Average Semantic Similarity (Measures meaning. Higher = better meaning preservation)
total_similarity = sum([calculate_similarity(orig, para) for orig, para in zip(input_texts, paraphrased_outputs)])
avg_similarity = total_similarity / len(input_texts)

print("1. Meaning Preservation (Semantic Similarity)")
print("   Metric: Sentence Transformer Cosine Similarity")
print(f"   Average Score: {avg_similarity:.4f} (Closer to 1.0 means meaning is perfectly preserved)\n")

print("2. Originality / Structural Overlap (ROUGE)")
print("   (Comparing paraphrase to original. Lower score means the model successfully changed the structure/words)")
print(f"   ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"   ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"   ROUGE-L: {rouge_results['rougeL']:.4f}\n")

print("3. Fluency & Grammar")
print("   Metric: language_tool_python rules triggered")
print("   Status: All outputs pass through a final correction layer to ensure 0 spelling/grammar errors upon output.")

--- EVALUATION REPORT ---

1. Meaning Preservation (Semantic Similarity)
   Metric: Sentence Transformer Cosine Similarity
   Average Score: 0.9873 (Closer to 1.0 means meaning is perfectly preserved)

2. Originality / Structural Overlap (ROUGE)
   (Comparing paraphrase to original. Lower score means the model successfully changed the structure/words)
   ROUGE-1: 0.9233
   ROUGE-2: 0.7805
   ROUGE-L: 0.8635

3. Fluency & Grammar
   Metric: language_tool_python rules triggered
   Status: All outputs pass through a final correction layer to ensure 0 spelling/grammar errors upon output.
